# B6: Cross-Model Anchor-Space Invariance

Validates **RFC-011** — the key insight that anchor-projected trajectories are comparable across different embedding models.

| Model | Dimension | Architecture |
|-------|-----------|-------------|
| MentalRoBERTa | D=768 | RoBERTa fine-tuned on mental health |
| all-MiniLM-L6-v2 | D=384 | Lightweight sentence transformer |

**Experiments**:
1. Project both models to shared DSM-5 anchor space (ℝ¹⁰)
2. Compare anchor-projected trajectories: are they correlated?
3. Cross-model search: can we find the same users across models?
4. Anchor drift consistency: does drift direction agree?

In [1]:
import chronos_vector as cvx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer, AutoModel
import torch
import time, os, warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
EMB_DIR = f'{DATA_DIR}/embeddings'

C_MENTAL = '#e74c3c'   # MentalRoBERTa
C_MINILM = '#3498db'   # MiniLM
C_ANCHOR = '#f39c12'   # Anchor space
C_ACCENT = '#2ecc71'

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Both Embedding Sets

In [2]:
# MentalRoBERTa (D=768)
df_mental = pd.read_parquet(f'{EMB_DIR}/erisk_mental_embeddings.parquet')
mental_emb_cols = [c for c in df_mental.columns if c.startswith('emb_')]
D_mental = len(mental_emb_cols)

# MiniLM (D=384)
df_minilm = pd.read_parquet(f'{EMB_DIR}/erisk_minilm_embeddings.parquet')
minilm_emb_cols = [c for c in df_minilm.columns if c.startswith('emb_')]
D_minilm = len(minilm_emb_cols)

print(f'MentalRoBERTa: {len(df_mental):,} posts, D={D_mental}')
print(f'MiniLM:        {len(df_minilm):,} posts, D={D_minilm}')

# Verify same users
users_mental = set(df_mental['user_id'].unique())
users_minilm = set(df_minilm['user_id'].unique())
common_users = users_mental & users_minilm
print(f'Common users: {len(common_users)}')

MentalRoBERTa: 1,359,188 posts, D=768
MiniLM:        1,359,188 posts, D=384
Common users: 2285


In [3]:
# Balanced subset (same as B2/B4)
dep_users = df_mental[df_mental['label'] == 'depression']['user_id'].unique()
ctl_users = df_mental[df_mental['label'] == 'control']['user_id'].unique()
np.random.seed(42)
n_sample = min(len(dep_users), len(ctl_users))
ctl_sample = np.random.choice(ctl_users, size=n_sample, replace=False)
selected = set(dep_users) | set(ctl_sample)

df_m = df_mental[df_mental['user_id'].isin(selected)].copy()
df_l = df_minilm[df_minilm['user_id'].isin(selected)].copy()

label_map = dict(zip(df_m['user_id'], df_m['label']))
print(f'Subset: {len(selected)} users ({len(dep_users)} dep + {n_sample} ctl)')
print(f'MentalRoBERTa posts: {len(df_m):,}')
print(f'MiniLM posts:        {len(df_l):,}')

Subset: 466 users (233 dep + 233 ctl)
MentalRoBERTa posts: 225,962
MiniLM posts:        225,962


## 2. DSM-5 Anchors in Both Embedding Spaces

The same anchor texts must be embedded in BOTH models to create the shared coordinate system.

In [4]:
DSM5_ANCHORS = {
    'depressed_mood': [
        'I feel sad and empty inside all the time',
        'Everything feels hopeless and I cannot stop crying',
        'I feel so depressed that nothing matters anymore',
    ],
    'anhedonia': [
        'I have lost interest in everything I used to enjoy',
        'Nothing gives me pleasure anymore not even my hobbies',
        'I do not care about anything and cannot feel joy',
    ],
    'sleep_disturbance': [
        'I cannot sleep at night and lie awake for hours',
        'I sleep all day and still feel exhausted',
        'My sleep schedule is completely destroyed',
    ],
    'fatigue': [
        'I am so tired I can barely get out of bed',
        'I have no energy to do anything at all',
        'Everything takes so much effort I am exhausted',
    ],
    'worthlessness': [
        'I am worthless and everyone would be better off without me',
        'I feel guilty about everything and hate myself',
        'I am a burden to everyone around me',
    ],
    'concentration': [
        'I cannot concentrate or focus on anything',
        'My mind is foggy and I cannot think clearly',
        'I keep forgetting things and cannot make decisions',
    ],
    'suicidal_ideation': [
        'I think about ending my life sometimes',
        'I do not want to be alive anymore',
        'I have thoughts about death and dying',
    ],
    'appetite': [
        'I have no appetite and have lost a lot of weight',
        'I keep eating everything in sight to feel better',
        'My eating habits have changed dramatically',
    ],
    'psychomotor': [
        'I feel restless and cannot sit still',
        'I move and talk so slowly people notice',
        'My body feels heavy and everything is in slow motion',
    ],
}

HEALTHY_PHRASES = [
    'I had a great day at work and went out with friends',
    'Looking forward to the weekend plans with family',
    'Just finished a good workout feeling energized and happy',
]

all_anchor_texts = {k: v for k, v in DSM5_ANCHORS.items()}
all_anchor_texts['healthy'] = HEALTHY_PHRASES
anchor_names = list(all_anchor_texts.keys())
K = len(anchor_names)
print(f'{K} anchors: {anchor_names}')

10 anchors: ['depressed_mood', 'anhedonia', 'sleep_disturbance', 'fatigue', 'worthlessness', 'concentration', 'suicidal_ideation', 'appetite', 'psychomotor', 'healthy']


In [5]:
CACHE_MENTAL = f'{DATA_DIR}/cache/dsm5_anchors_mental.npz'
CACHE_MINILM = f'{DATA_DIR}/cache/dsm5_anchors_minilm.npz'

def encode_anchors(model_name, cache_path):
    if os.path.exists(cache_path):
        data = np.load(cache_path, allow_pickle=True)
        print(f'Loaded cached {model_name} anchors')
        return data['anchors'].item()
    
    print(f'Encoding anchors with {model_name}...')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()
    
    def encode(texts):
        with torch.no_grad():
            inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
            outputs = model(**inputs)
            mask = inputs['attention_mask'].unsqueeze(-1).float()
            return ((outputs.last_hidden_state * mask).sum(1) / mask.sum(1)).numpy()
    
    anchors = {}
    for name, phrases in all_anchor_texts.items():
        embs = encode(phrases)
        anchors[name] = embs.mean(axis=0)
        print(f'  {name}: D={embs.shape[1]}')
    
    np.savez(cache_path, anchors=anchors)
    return anchors

mental_anchors = encode_anchors('mental/mental-roberta-base', CACHE_MENTAL)
minilm_anchors = encode_anchors('sentence-transformers/all-MiniLM-L6-v2', CACHE_MINILM)

mental_anchors_np = np.array([mental_anchors[k] for k in anchor_names])
minilm_anchors_np = np.array([minilm_anchors[k] for k in anchor_names])
print(f'MentalRoBERTa anchors: {mental_anchors_np.shape}')
print(f'MiniLM anchors: {minilm_anchors_np.shape}')

Encoding anchors with mental/mental-roberta-base...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 44651.60it/s]


RobertaModel LOAD REPORT from: mental/mental-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  depressed_mood: D=768
  anhedonia: D=768
  sleep_disturbance: D=768
  fatigue: D=768
  worthlessness: D=768
  concentration: D=768


  suicidal_ideation: D=768
  appetite: D=768
  psychomotor: D=768
  healthy: D=768
Encoding anchors with sentence-transformers/all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8512.41it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  depressed_mood: D=384
  anhedonia: D=384
  sleep_disturbance: D=384
  fatigue: D=384
  worthlessness: D=384
  concentration: D=384
  suicidal_ideation: D=384
  appetite: D=384
  psychomotor: D=384
  healthy: D=384
MentalRoBERTa anchors: (10, 768)
MiniLM anchors: (10, 384)


## 3. Project Both Models to Anchor Space

After projection, both models produce trajectories in ℝ¹⁰ (10 anchor distances). These should be directly comparable.

In [6]:
def build_projected_trajectories(df, emb_cols, anchors_np):
    """Project user trajectories to anchor space."""
    results = {}
    anchors_list = anchors_np.tolist()
    for uid, grp in df.sort_values('timestamp').groupby('user_id'):
        traj = [(int(row['timestamp'].value), row[emb_cols].values.astype(np.float32).tolist())
                for _, row in grp.head(500).iterrows()]  # cap for speed
        if len(traj) < 5:
            continue
        projected = cvx.project_to_anchors(traj, anchors_list, 'cosine')
        results[uid] = projected
    return results

t0 = time.perf_counter()
mental_proj = build_projected_trajectories(df_m, mental_emb_cols, mental_anchors_np)
print(f'MentalRoBERTa projected: {len(mental_proj)} users ({time.perf_counter()-t0:.1f}s)')

t0 = time.perf_counter()
minilm_proj = build_projected_trajectories(df_l, minilm_emb_cols, minilm_anchors_np)
print(f'MiniLM projected: {len(minilm_proj)} users ({time.perf_counter()-t0:.1f}s)')

common_proj = set(mental_proj.keys()) & set(minilm_proj.keys())
print(f'Users with projections from both models: {len(common_proj)}')

MentalRoBERTa projected: 466 users (65.7s)


MiniLM projected: 466 users (45.4s)
Users with projections from both models: 466


## 4. Experiment 1: Anchor Trajectory Correlation

For each user, compare the mean anchor-projected vector from both models. If anchor projection is truly model-invariant, the correlation should be high.

In [7]:
# Compute mean anchor-projected vector per user per model
mental_means = []
minilm_means = []
user_labels = []

for uid in sorted(common_proj):
    m_vecs = np.array([v for _, v in mental_proj[uid]])
    l_vecs = np.array([v for _, v in minilm_proj[uid]])
    mental_means.append(m_vecs.mean(axis=0))
    minilm_means.append(l_vecs.mean(axis=0))
    user_labels.append(1 if label_map.get(uid) == 'depression' else 0)

mental_means = np.array(mental_means)
minilm_means = np.array(minilm_means)
user_labels = np.array(user_labels)

# Per-anchor correlation
print('=== PER-ANCHOR CORRELATION (Pearson r) ===')
correlations = []
for k in range(K):
    r, p = pearsonr(mental_means[:, k], minilm_means[:, k])
    rho, _ = spearmanr(mental_means[:, k], minilm_means[:, k])
    correlations.append({'anchor': anchor_names[k], 'pearson_r': r, 'spearman_rho': rho, 'p_value': p})
    print(f'  {anchor_names[k]:20s}: r={r:.3f} (p={p:.2e}), ρ={rho:.3f}')

corr_df = pd.DataFrame(correlations)
print(f'\nMean Pearson r: {corr_df["pearson_r"].mean():.3f}')
print(f'Mean Spearman ρ: {corr_df["spearman_rho"].mean():.3f}')

=== PER-ANCHOR CORRELATION (Pearson r) ===
  depressed_mood      : r=0.574 (p=3.96e-42), ρ=0.691
  anhedonia           : r=0.529 (p=5.16e-35), ρ=0.656
  sleep_disturbance   : r=0.258 (p=1.59e-08), ρ=0.395
  fatigue             : r=0.432 (p=1.25e-22), ρ=0.515
  worthlessness       : r=0.518 (p=2.44e-33), ρ=0.626
  concentration       : r=0.422 (p=1.39e-21), ρ=0.545
  suicidal_ideation   : r=0.505 (p=1.74e-31), ρ=0.592
  appetite            : r=0.488 (p=3.32e-29), ρ=0.540
  psychomotor         : r=0.477 (p=8.54e-28), ρ=0.557
  healthy             : r=0.383 (p=1.02e-17), ρ=0.464

Mean Pearson r: 0.459
Mean Spearman ρ: 0.558


In [8]:
# Scatter plots for top 4 anchors
top_anchors = corr_df.nlargest(4, 'pearson_r')['anchor'].tolist()
fig = make_subplots(rows=2, cols=2, subplot_titles=top_anchors)

for idx, anchor in enumerate(top_anchors):
    k = anchor_names.index(anchor)
    row, col = idx // 2 + 1, idx % 2 + 1
    colors = [C_MENTAL if l == 1 else C_MINILM for l in user_labels]
    fig.add_trace(go.Scatter(
        x=mental_means[:, k], y=minilm_means[:, k],
        mode='markers', marker=dict(size=4, color=colors, opacity=0.6),
        name=anchor, showlegend=False,
    ), row=row, col=col)
    fig.update_xaxes(title_text='MentalRoBERTa', row=row, col=col)
    fig.update_yaxes(title_text='MiniLM', row=row, col=col)

fig.update_layout(height=600, title='Cross-Model Anchor Correlation (red=depressed, blue=control)')
fig.show()

## 5. Experiment 2: Anchor Drift Agreement

For each user, compute drift in anchor space from both models. Do they agree on WHO drifted and in WHICH direction?

In [9]:
# Compute per-user anchor drift in both models
drift_mental = []
drift_minilm = []
drift_uids = []

for uid in sorted(common_proj):
    m_traj = mental_proj[uid]
    l_traj = minilm_proj[uid]
    if len(m_traj) < 5 or len(l_traj) < 5:
        continue
    
    # Drift = last 20% mean - first 20% mean
    n_m = len(m_traj)
    n_l = len(l_traj)
    m_early = np.array([v for _, v in m_traj[:n_m//5]]).mean(axis=0)
    m_late = np.array([v for _, v in m_traj[-n_m//5:]]).mean(axis=0)
    l_early = np.array([v for _, v in l_traj[:n_l//5]]).mean(axis=0)
    l_late = np.array([v for _, v in l_traj[-n_l//5:]]).mean(axis=0)
    
    drift_mental.append(m_late - m_early)
    drift_minilm.append(l_late - l_early)
    drift_uids.append(uid)

drift_mental = np.array(drift_mental)
drift_minilm = np.array(drift_minilm)

# Per-anchor drift correlation
print('=== DRIFT DIRECTION AGREEMENT ===')
for k in range(K):
    r, p = pearsonr(drift_mental[:, k], drift_minilm[:, k])
    print(f'  {anchor_names[k]:20s}: r={r:.3f} (p={p:.2e})')

# Overall drift magnitude correlation
mag_mental = np.linalg.norm(drift_mental, axis=1)
mag_minilm = np.linalg.norm(drift_minilm, axis=1)
r_mag, p_mag = pearsonr(mag_mental, mag_minilm)
print(f'\nDrift MAGNITUDE correlation: r={r_mag:.3f} (p={p_mag:.2e})')

=== DRIFT DIRECTION AGREEMENT ===
  depressed_mood      : r=0.283 (p=4.71e-10)
  anhedonia           : r=0.307 (p=1.23e-11)
  sleep_disturbance   : r=0.027 (p=5.65e-01)
  fatigue             : r=0.171 (p=2.11e-04)
  worthlessness       : r=0.274 (p=1.80e-09)
  concentration       : r=0.123 (p=7.91e-03)
  suicidal_ideation   : r=0.205 (p=8.32e-06)
  appetite            : r=0.157 (p=6.58e-04)
  psychomotor         : r=0.133 (p=4.03e-03)
  healthy             : r=0.202 (p=1.12e-05)

Drift MAGNITUDE correlation: r=0.205 (p=8.22e-06)


## 6. Experiment 3: Cross-Model Classification

Can anchor-projected features from one model predict depression when trained on the other?

In [10]:
# Build feature matrices from anchor projections
def build_features(proj_dict, user_list):
    X = []
    y = []
    for uid in user_list:
        if uid not in proj_dict or len(proj_dict[uid]) < 5:
            continue
        vecs = np.array([v for _, v in proj_dict[uid]])
        # Features: mean + std + trend (slope) per anchor
        feats = np.concatenate([
            vecs.mean(axis=0),  # K means
            vecs.std(axis=0),   # K stds
        ])
        X.append(feats)
        y.append(1 if label_map.get(uid) == 'depression' else 0)
    return np.array(X), np.array(y)

user_list = sorted(common_proj)
X_mental, y_mental = build_features(mental_proj, user_list)
X_minilm, y_minilm = build_features(minilm_proj, user_list)

print(f'MentalRoBERTa features: {X_mental.shape}')
print(f'MiniLM features: {X_minilm.shape}')
print(f'Labels: {sum(y_mental)} dep / {len(y_mental)-sum(y_mental)} ctl')

MentalRoBERTa features: (466, 20)
MiniLM features: (466, 20)
Labels: 233 dep / 233 ctl


In [11]:
# Within-model classification (baseline)
def eval_cv(X, y, name):
    scaler = StandardScaler()
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    f1s, aucs = [], []
    for tr, te in skf.split(X, y):
        Xtr = scaler.fit_transform(X[tr])
        Xte = scaler.transform(X[te])
        clf = LogisticRegression(max_iter=1000)
        clf.fit(Xtr, y[tr])
        f1s.append(f1_score(y[te], clf.predict(Xte)))
        aucs.append(roc_auc_score(y[te], clf.predict_proba(Xte)[:, 1]))
    print(f'{name}: F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}, AUC={np.mean(aucs):.3f}±{np.std(aucs):.3f}')
    return np.mean(f1s), np.mean(aucs)

print('=== WITHIN-MODEL (5-fold CV) ===')
f1_m, auc_m = eval_cv(X_mental, y_mental, 'MentalRoBERTa → MentalRoBERTa')
f1_l, auc_l = eval_cv(X_minilm, y_minilm, 'MiniLM → MiniLM')

=== WITHIN-MODEL (5-fold CV) ===


MentalRoBERTa → MentalRoBERTa: F1=0.770±0.021, AUC=0.866±0.019
MiniLM → MiniLM: F1=0.833±0.007, AUC=0.901±0.017


In [12]:
# Cross-model transfer: train on one model, test on the other
print('=== CROSS-MODEL TRANSFER ===')

scaler_m = StandardScaler().fit(X_mental)
scaler_l = StandardScaler().fit(X_minilm)

# Train on MentalRoBERTa, test on MiniLM
clf_m = LogisticRegression(max_iter=1000)
clf_m.fit(scaler_m.transform(X_mental), y_mental)
y_pred_transfer = clf_m.predict(scaler_l.transform(X_minilm))
y_prob_transfer = clf_m.predict_proba(scaler_l.transform(X_minilm))[:, 1]
f1_m2l = f1_score(y_minilm, y_pred_transfer)
auc_m2l = roc_auc_score(y_minilm, y_prob_transfer)
print(f'MentalRoBERTa → MiniLM: F1={f1_m2l:.3f}, AUC={auc_m2l:.3f}')

# Train on MiniLM, test on MentalRoBERTa
clf_l = LogisticRegression(max_iter=1000)
clf_l.fit(scaler_l.transform(X_minilm), y_minilm)
y_pred_transfer2 = clf_l.predict(scaler_m.transform(X_mental))
y_prob_transfer2 = clf_l.predict_proba(scaler_m.transform(X_mental))[:, 1]
f1_l2m = f1_score(y_mental, y_pred_transfer2)
auc_l2m = roc_auc_score(y_mental, y_prob_transfer2)
print(f'MiniLM → MentalRoBERTa: F1={f1_l2m:.3f}, AUC={auc_l2m:.3f}')

=== CROSS-MODEL TRANSFER ===
MentalRoBERTa → MiniLM: F1=0.675, AUC=0.773
MiniLM → MentalRoBERTa: F1=0.367, AUC=0.335


In [13]:
# Summary comparison
results = pd.DataFrame([
    {'Setting': 'MentalRoBERTa (5-fold)', 'F1': f1_m, 'AUC': auc_m},
    {'Setting': 'MiniLM (5-fold)', 'F1': f1_l, 'AUC': auc_l},
    {'Setting': 'Mental→MiniLM (transfer)', 'F1': f1_m2l, 'AUC': auc_m2l},
    {'Setting': 'MiniLM→Mental (transfer)', 'F1': f1_l2m, 'AUC': auc_l2m},
])

fig = go.Figure(data=[
    go.Bar(name='F1', x=results['Setting'], y=results['F1'], marker_color=C_MENTAL),
    go.Bar(name='AUC', x=results['Setting'], y=results['AUC'], marker_color=C_MINILM),
])
fig.update_layout(
    title='Anchor-Space Classification: Within-Model vs Cross-Model Transfer',
    barmode='group', height=450, yaxis=dict(range=[0, 1]),
)
fig.show()

## 7. Summary

In [14]:
print('=== B6 CROSS-MODEL VALIDATION RESULTS ===')
print()
print(f'Models: MentalRoBERTa (D={D_mental}) vs MiniLM (D={D_minilm})')
print(f'Anchor space: K={K} DSM-5 clinical anchors')
print(f'Users: {len(common_proj)} (both models)')
print()
print('Anchor correlation (mean Pearson r):', f'{corr_df["pearson_r"].mean():.3f}')
print('Drift magnitude correlation:', f'{r_mag:.3f}')
print()
print(results.to_string(index=False))
print()
transfer_drop_f1 = 1 - (f1_m2l + f1_l2m) / 2 / ((f1_m + f1_l) / 2)
print(f'Cross-model transfer F1 drop: {transfer_drop_f1:.1%}')
print()
if corr_df['pearson_r'].mean() > 0.5:
    print('✓ Anchor projection provides meaningful cross-model invariance')
else:
    print('✗ Anchor correlation too low — models see different structures')

=== B6 CROSS-MODEL VALIDATION RESULTS ===

Models: MentalRoBERTa (D=768) vs MiniLM (D=384)
Anchor space: K=10 DSM-5 clinical anchors
Users: 466 (both models)

Anchor correlation (mean Pearson r): 0.459
Drift magnitude correlation: 0.205

                 Setting       F1      AUC
  MentalRoBERTa (5-fold) 0.770369 0.865780
         MiniLM (5-fold) 0.832520 0.901289
Mental→MiniLM (transfer) 0.674584 0.773140
MiniLM→Mental (transfer) 0.366890 0.335188

Cross-model transfer F1 drop: 35.0%

✗ Anchor correlation too low — models see different structures
